# Live Demo 1 — Object Detection: fine-tuning Faster R-CNN on Penn-Fudan pedestrians

**Goal of this demo (≈ 60–75 min incl. discussion):** take a detector that was pre-trained on COCO (80 object classes),
replace its classification head with a *2-class* head (background + person) and fine-tune it on a tiny dataset
(170 images) — **on a laptop CPU, live**.

| Step | What students see | Approx. time (measured on the reference laptop, see README) |
|---|---|---|
| 1 | Dataset: masks → bounding boxes, visualised | < 10 s |
| 2 | Model: Faster R-CNN + MobileNetV3-Large + FPN, head replacement | < 5 s |
| 3 | Baseline: how good is COCO "person" *before* fine-tuning? | ≈ 15 s |
| 4 | Training loop with per-iteration loss prints | ≈ 10 min (or ~1 s with `USE_CACHED=True`) |
| 5 | Evaluation: our own AP@0.5 implementation, PR curves | ≈ 15 s |
| 6 | Predictions on held-out images: before vs after | ≈ 10 s |

> **Trainer note — safety net.** `USE_CACHED = True` (default) loads `checkpoints/fasterrcnn_mbv3_pennfudan.pth`
> if it exists, so the demo cannot fail live. Set it to `False` to train in front of the class (≈ 10 min).
> Good practice: start the training cell, then explain the architecture slides / README talking points while it runs.

## 0. Setup
Everything runs on CPU. Data lives in `../data` (downloaded the night before by `prepare_demo.py`).

In [ ]:
from pathlib import Path
import time, math, random, copy

import numpy as np
import torch, torchvision
from torch.utils.data import Dataset, Subset, DataLoader
from torchvision.ops import box_iou
import torchvision.transforms.functional as TF
from torchvision.models.detection import (fasterrcnn_mobilenet_v3_large_fpn,
                                          FasterRCNN_MobileNet_V3_Large_FPN_Weights)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

USE_CACHED = True            # <- set False to train live (~10 min on a modern laptop CPU)
SEED = 42

DATA_DIR = Path("..") / "data" / "PennFudanPed"
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR  = Path("outputs");     OUT_DIR.mkdir(exist_ok=True)
CKPT_PATH = CKPT_DIR / "fasterrcnn_mbv3_pennfudan.pth"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | threads {torch.get_num_threads()}")
assert DATA_DIR.exists(), "Run `python prepare_demo.py` first (downloads Penn-Fudan into ../data)"
print("checkpoint exists:", CKPT_PATH.exists(), "| USE_CACHED =", USE_CACHED)

## 1. The dataset: Penn-Fudan pedestrians

* 170 street images (Univ. of Pennsylvania + Fudan University), 345 annotated pedestrians.
* Annotation is an **instance mask** PNG: pixel value 0 = background, 1 = first person, 2 = second person, …
* A detector needs **boxes** `[x1, y1, x2, y2]` → we derive the tightest box around each instance mask
  (exactly what the official torchvision tutorial does).

**Target format expected by torchvision detection models** (one dict per image):
`boxes` (FloatTensor[N, 4], absolute pixel coords, x1<x2, y1<y2) and `labels` (Int64Tensor[N]; 0 is reserved for background).

**Speaker note:** the model does its own resizing & normalisation inside (`GeneralizedRCNNTransform`), so the dataset only
returns a `[0,1]` float tensor. Augmentation = random horizontal flip — remember to flip the *boxes* too!

In [ ]:
class PennFudanDataset(Dataset):
    # Returns (image_tensor[3,H,W] in [0,1], target_dict) - the format torchvision detectors expect.
    def __init__(self, root, train: bool):
        self.root, self.train = Path(root), train
        self.imgs  = sorted((self.root / "PNGImages").glob("*.png"))
        self.masks = sorted((self.root / "PedMasks").glob("*.png"))
        assert len(self.imgs) == len(self.masks) == 170, "Penn-Fudan incomplete - rerun prepare_demo.py"
        for i, m in zip(self.imgs, self.masks):
            assert m.stem == i.stem + "_mask"
        # masks -> boxes once (cheap, and keeps __getitem__ fast)
        self.boxes = [self.boxes_from_mask(m) for m in self.masks]

    @staticmethod
    def boxes_from_mask(mask_path):
        mask = np.array(Image.open(mask_path))
        obj_ids = np.unique(mask)
        obj_ids = obj_ids[obj_ids != 0]            # 0 = background
        boxes = []
        for oid in obj_ids:
            ys, xs = np.where(mask == oid)
            x1, y1, x2, y2 = xs.min(), ys.min(), xs.max() + 1, ys.max() + 1
            if (x2 - x1) > 1 and (y2 - y1) > 1:    # drop degenerate boxes
                boxes.append([x1, y1, x2, y2])
        return torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = TF.pil_to_tensor(Image.open(self.imgs[idx]).convert("RGB")).float() / 255.0
        boxes = self.boxes[idx].clone()
        if self.train and random.random() < 0.5:   # horizontal flip: image AND boxes
            img = img.flip(-1)
            W = img.shape[-1]
            boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
        target = {"boxes": boxes,
                  "labels": torch.ones(len(boxes), dtype=torch.int64),   # 1 = person
                  "image_id": idx}
        return img, target

# fixed, reproducible split: 120 train / 50 held-out test images
perm = torch.randperm(170, generator=torch.Generator().manual_seed(SEED)).tolist()
train_idx, test_idx = perm[:120], perm[120:]
ds_train = Subset(PennFudanDataset(DATA_DIR, train=True),  train_idx)
ds_test  = Subset(PennFudanDataset(DATA_DIR, train=False), test_idx)

n_train_boxes = sum(len(ds_train.dataset.boxes[i]) for i in train_idx)
n_test_boxes  = sum(len(ds_test.dataset.boxes[i])  for i in test_idx)
print(f"train: {len(ds_train)} images / {n_train_boxes} people   test: {len(ds_test)} images / {n_test_boxes} people")
img, tgt = ds_test[0]
print("image tensor:", tuple(img.shape), img.dtype, "| target:", {k: (tuple(v.shape) if torch.is_tensor(v) else v) for k, v in tgt.items()})

In [ ]:
def draw_boxes(ax, img, boxes, color="lime", scores=None, title=None, lw=2):
    ax.imshow(img.permute(1, 2, 0).numpy()); ax.axis("off")
    for i, b in enumerate(boxes.tolist()):
        x1, y1, x2, y2 = b
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=lw))
        if scores is not None:
            ax.text(x1, y1 - 3, f"{float(scores[i]):.2f}", color="white", fontsize=8,
                    bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor="none"))
    if title: ax.set_title(title, fontsize=10)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
full_train = PennFudanDataset(DATA_DIR, train=False)          # no flip, for display
for ax, i in zip(axes, train_idx[:4]):
    im, t = full_train[i]
    m = np.array(Image.open(full_train.masks[i]))
    draw_boxes(ax, im, t["boxes"], title=f"{full_train.imgs[i].name}: {len(t['boxes'])} person(s)")
    ax.imshow(np.ma.masked_equal(m, 0), alpha=0.35, cmap="tab10")    # overlay the instance masks
plt.suptitle("Penn-Fudan: instance masks (colour) -> derived bounding boxes (green)")
plt.tight_layout(); plt.savefig(OUT_DIR / "det_dataset_samples.png", dpi=90); plt.show()

## 2. The model: Faster R-CNN with a MobileNetV3-Large + FPN backbone

```
image ─► GeneralizedRCNNTransform (resize so short side = 800, normalise)
      ─► Backbone: MobileNetV3-Large (ImageNet → COCO pre-trained)
      ─► FPN: multi-scale feature maps (small + large objects)
      ─► RPN (Region Proposal Network): anchors → objectness + box deltas → NMS → ~1000 proposals
      ─► RoIAlign 7×7 per proposal ─► TwoMLPHead (fc6: 12544→1024, fc7: 1024→1024)
      ─► FastRCNNPredictor: class scores (C) + box regression (4·C)   ◄── the only part we REPLACE
      ─► post-processing: score threshold, per-class NMS, top-100
```

**Why this variant?** `fasterrcnn_mobilenet_v3_large_fpn` (COCO box mAP 32.8) is ~19 M parameters and runs
≈ 0.2–0.3 s/image on a laptop CPU — the ResNet-50 version is ~5× slower, which is too slow for a live CPU demo.

**Transfer learning recipe:** keep the COCO-pretrained backbone + RPN + box head, swap the final predictor
(91 COCO classes → 2 classes). The new predictor starts random; everything else starts from very good features.
We train with **multi-scale** input sizes (short side randomly 480…800), which makes the model robust to the input
resolution changes we will try in the optimisation notebook.

In [ ]:
WEIGHTS = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT     # COCO-pretrained (cached by prepare_demo.py)
TRAIN_SIZES = (480, 544, 608, 672, 736, 800)                     # multi-scale training; eval uses the last (800)

def build_finetune_model(num_classes=2):
    model = fasterrcnn_mobilenet_v3_large_fpn(weights=WEIGHTS, min_size=TRAIN_SIZES, max_size=1333)
    in_features = model.roi_heads.box_predictor.cls_score.in_features   # 1024
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = build_finetune_model()
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parameters: {n_total/1e6:.2f} M total, {n_train/1e6:.2f} M trainable "
      f"(first backbone blocks are frozen by torchvision's default trainable_backbone_layers=3)")
print(model.roi_heads.box_predictor)

## 3. Evaluation metric: AP@0.5, implemented from scratch

1. **IoU** (Intersection over Union) between a predicted and a ground-truth box: area(A∩B) / area(A∪B).
2. Sort all detections (over the whole test set) by confidence.
3. Walk down the list: a detection is a **true positive** if it overlaps a *not-yet-matched* GT box with IoU ≥ 0.5,
   otherwise a **false positive** (duplicates count as FP — that is why NMS matters).
4. This gives a **precision–recall curve**; **AP** = area under the (monotone-envelope) PR curve (VOC all-point interpolation).
5. **mAP** = mean of AP over classes (we have one class) — COCO additionally averages over IoU thresholds 0.50:0.95.

In [ ]:
@torch.inference_mode()
def run_inference(model, dataset, keep_label=None):
    # Returns list of prediction dicts + list of GT boxes. keep_label filters one class (e.g. COCO 'person' = 1).
    model.eval()
    preds, gts = [], []
    for img, tgt in dataset:
        out = model([img])[0]
        if keep_label is not None:
            k = out["labels"] == keep_label
            out = {kk: v[k] for kk, v in out.items()}
        preds.append(out); gts.append(tgt["boxes"])
    return preds, gts

def average_precision(preds, gts, iou_thr=0.5):
    # VOC-style all-point-interpolated AP for a single class. Returns (AP, precision[], recall[]).
    n_gt = sum(len(g) for g in gts)
    all_scores, all_tp = [], []
    for p, g in zip(preds, gts):
        order = p["scores"].argsort(descending=True)
        boxes, scores = p["boxes"][order], p["scores"][order]
        matched = torch.zeros(len(g), dtype=torch.bool)
        ious = box_iou(boxes, g) if (len(boxes) and len(g)) else torch.zeros(len(boxes), len(g))
        for i in range(len(boxes)):
            is_tp = False
            if len(g):
                row = ious[i].clone(); row[matched] = -1.0     # each GT can be matched only once
                j = int(row.argmax())
                if row[j] >= iou_thr:
                    matched[j] = True; is_tp = True
            all_scores.append(float(scores[i])); all_tp.append(is_tp)
    if not all_scores:
        return 0.0, np.array([0.0]), np.array([0.0])
    order = np.argsort(-np.asarray(all_scores), kind="stable")
    tp = np.asarray(all_tp, dtype=float)[order]
    tp_cum, fp_cum = np.cumsum(tp), np.cumsum(1 - tp)
    recall = tp_cum / max(n_gt, 1)
    precision = tp_cum / (tp_cum + fp_cum)
    mrec = np.concatenate([[0.0], recall, [1.0]])
    mpre = np.concatenate([[0.0], precision, [0.0]])
    for i in range(len(mpre) - 2, -1, -1):                   # monotone envelope
        mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))
    return ap, precision, recall

def operating_point(preds, gts, score_thr=0.5, iou_thr=0.5):
    # What a user of the deployed detector sees: only boxes with score >= score_thr are shown.
    tp = fp = 0
    for p, g in zip(preds, gts):
        keep = p["scores"] >= score_thr
        boxes = p["boxes"][keep][p["scores"][keep].argsort(descending=True)]
        matched = torch.zeros(len(g), dtype=torch.bool)
        ious = box_iou(boxes, g) if (len(boxes) and len(g)) else torch.zeros(len(boxes), len(g))
        for i in range(len(boxes)):
            if len(g):
                row = ious[i].clone(); row[matched] = -1.0; j = int(row.argmax())
                if row[j] >= iou_thr:
                    matched[j] = True; tp += 1; continue
            fp += 1
    n_gt = sum(len(g) for g in gts)
    return {"TP": tp, "FP": fp, "FN": n_gt - tp,
            "precision": tp / max(tp + fp, 1), "recall": tp / max(n_gt, 1)}

def evaluate(model, dataset, keep_label=None, verbose=True):
    t0 = time.perf_counter()
    preds, gts = run_inference(model, dataset, keep_label)
    ap50, prec, rec = average_precision(preds, gts, 0.5)
    ap_coco = float(np.mean([average_precision(preds, gts, t)[0] for t in np.arange(0.5, 0.96, 0.05)]))
    op = operating_point(preds, gts)
    res = {"AP50": ap50, "AP50_95": ap_coco, "max_recall": float(rec.max()), "op": op,
           "prec": prec, "rec": rec, "preds": preds, "gts": gts, "eval_s": time.perf_counter() - t0}
    if verbose:
        print(f"AP@0.5 = {ap50:.3f} | AP@[.5:.95] = {ap_coco:.3f} | at score>=0.5: precision {op['precision']:.3f}, "
              f"recall {op['recall']:.3f} (TP {op['TP']}, FP {op['FP']}, FN {op['FN']}) "
              f"| {len(dataset)} images in {res['eval_s']:.1f}s")
    return res

# sanity check of the metric: perfect predictions must give AP = 1.0
fake = [{"boxes": g, "scores": torch.ones(len(g)), "labels": torch.ones(len(g), dtype=torch.int64)}
        for g in [ds_test.dataset.boxes[i] for i in test_idx]]
print("AP of perfect predictions:", average_precision(fake, [ds_test.dataset.boxes[i] for i in test_idx])[0])

## 4. Baselines *before* fine-tuning

* **(a) COCO-pretrained model, class "person" only** — COCO already contains ~250k people, so this is a strong baseline!
  Its mistakes are mostly *annotation-style* mismatches (Penn-Fudan does not label every tiny/background/occluded person,
  COCO boxes are drawn a bit differently) — fine-tuning adapts to **this dataset's definition** of a pedestrian.
* **(b) Our 2-class model right after replacing the head** — the new predictor is random, so it is useless until trained.

**Question to the class:** if the pre-trained model is already good, why fine-tune at all? (Answer: new classes that COCO
does not have, dataset-specific annotation conventions, domain shift — e.g. thermal/medical/drone images, and to get a
smaller, specialised 2-class model.)

**Two views of quality** are printed for each model:
* **AP** summarises the whole precision–recall curve (all score thresholds) — a *ranking* metric.
* **Precision / recall at score ≥ 0.5** is what an end user of the deployed detector actually sees (boxes drawn on screen):
  FP = boxes on things that are not annotated pedestrians, FN = missed pedestrians.

**Be honest with the class:** "person" is COCO's most frequent class, so the COCO model's AP@0.5 on Penn-Fudan is already
~0.98 — fine-tuning cannot raise AP@0.5 much here. What fine-tuning *does* show: (1) a brand-new random head goes from
AP ≈ 0.02 to ≈ 0.96 in **one epoch** on 120 images, (2) the fine-tuned model learns Penn-Fudan's *definition* of a pedestrian
(e.g. it stops firing confidently on people cut off at the image border, which Penn-Fudan does not annotate) → fewer
false positives at the operating threshold. For a class COCO does *not* contain (e.g. "hard hat", "tumour", "solar panel")
the pre-trained baseline would be ~0 and fine-tuning is the only option.

In [ ]:
coco_model = fasterrcnn_mobilenet_v3_large_fpn(weights=WEIGHTS)     # 91 COCO classes, label 1 = person
print("(a) COCO-pretrained, 'person' class only:")
res_coco = evaluate(coco_model, ds_test, keep_label=1)
print("(b) new 2-class head, NOT trained yet:")
res_random_head = evaluate(model, ds_test)

## 5. Fine-tuning (the live part)

* **Optimizer:** SGD, lr 0.01, momentum 0.9, weight decay 5e-4, batch size 4 (the torchvision reference recipe, scaled).
* **Schedule:** linear warm-up for 30 iterations, then cosine decay — warm-up avoids blowing up the fresh random head.
* In `train()` mode the model **returns a dict of losses** instead of predictions:
  * `loss_objectness` – RPN: is there an object in this anchor? (binary)
  * `loss_rpn_box_reg` – RPN: refine anchor → proposal
  * `loss_classifier` – box head: person vs background for each sampled proposal
  * `loss_box_reg` – box head: refine proposal → final box
* After each epoch we evaluate AP@0.5 on the 50 held-out images.

**Speaker note while it runs:** walk through the anchors/RPN/NMS talking points in the README. Point out that the loss
drops fast because the backbone already "knows" what people look like.

In [ ]:
EPOCHS, BATCH_SIZE, LR, WARMUP_ITERS, PRINT_EVERY = 6, 4, 0.01, 30, 5

def collate(batch):
    return tuple(zip(*batch))

def train_detector(model):
    g = torch.Generator().manual_seed(SEED)
    loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                        num_workers=0, generator=g)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=5e-4)
    total_iters = EPOCHS * len(loader)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda it: min(1.0, (it + 1) / WARMUP_ITERS) * 0.5 * (1 + math.cos(math.pi * it / total_iters)))
    history = {"iter_loss": [], "epoch_ap50": [], "epoch_time_s": []}
    t_start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        t_ep = time.perf_counter()
        for it, (imgs, targets) in enumerate(loader, 1):
            t_it = time.perf_counter()
            loss_dict = model(list(imgs), list(targets))
            loss = sum(loss_dict.values())
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            history["iter_loss"].append({k: float(v) for k, v in loss_dict.items()} | {"total": float(loss)})
            if it % PRINT_EVERY == 0 or it == 1:
                parts = " ".join(f"{k.replace('loss_', '')}={float(v):.3f}" for k, v in loss_dict.items())
                print(f"[epoch {epoch}/{EPOCHS} it {it:2d}/{len(loader)}] loss={float(loss):.3f} ({parts}) "
                      f"lr={opt.param_groups[0]['lr']:.4f} {time.perf_counter() - t_it:.2f}s/it", flush=True)
        train_s = time.perf_counter() - t_ep
        res = evaluate(model, ds_test, verbose=False)
        history["epoch_ap50"].append(res["AP50"]); history["epoch_time_s"].append(train_s)
        print(f"==> epoch {epoch} done: train {train_s:.0f}s | held-out AP@0.5 = {res['AP50']:.3f} "
              f"(eval {res['eval_s']:.0f}s)", flush=True)
    history["total_time_s"] = time.perf_counter() - t_start
    return history

if USE_CACHED and CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
    model.load_state_dict(ckpt["model"])
    history = ckpt["history"]
    print(f"Loaded cached checkpoint {CKPT_PATH} (originally trained in {history['total_time_s']/60:.1f} min).")
else:
    print(f"Training for {EPOCHS} epochs on CPU ...")
    history = train_detector(model)
    torch.save({"model": model.state_dict(), "history": history, "train_sizes": TRAIN_SIZES,
                "num_classes": 2, "split_seed": SEED, "arch": "fasterrcnn_mobilenet_v3_large_fpn"}, CKPT_PATH)
    print(f"Total training time {history['total_time_s']/60:.1f} min -> saved {CKPT_PATH} "
          f"({CKPT_PATH.stat().st_size/1e6:.1f} MB)")

In [ ]:
losses = history["iter_loss"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for k in ["total", "loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"]:
    v = np.array([d[k] for d in losses])
    smooth = np.convolve(v, np.ones(5) / 5, mode="valid")
    axes[0].plot(smooth, label=k)
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("loss (5-iter moving avg)"); axes[0].legend(); axes[0].set_title("Training losses")
axes[1].plot(range(1, len(history["epoch_ap50"]) + 1), history["epoch_ap50"], "o-")
axes[1].axhline(res_coco["AP50"], color="gray", ls="--", label="COCO-pretrained 'person' (no fine-tuning)")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("held-out AP@0.5"); axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].set_title("Held-out AP@0.5 per epoch")
plt.tight_layout(); plt.savefig(OUT_DIR / "det_training_curves.png", dpi=90); plt.show()

## 6. Evaluation after fine-tuning

In [ ]:
print("Fine-tuned 2-class model:")
res_ft = evaluate(model, ds_test)

import pandas as pd
def summary_row(name, r):
    op = r["op"]
    return {"model": name, "AP@0.5": r["AP50"], "AP@[.5:.95]": r["AP50_95"],
            "precision@0.5": op["precision"], "recall@0.5": op["recall"],
            "FP@0.5": op["FP"], "FN@0.5": op["FN"]}
summary = pd.DataFrame([
    summary_row("COCO-pretrained, 'person' only (no fine-tuning)", res_coco),
    summary_row("2-class head, before training", res_random_head),
    summary_row(f"fine-tuned ({len(history['epoch_ap50'])} epochs, 120 images)", res_ft),
]).set_index("model").round(3)
display(summary)

plt.figure(figsize=(6, 5))
for r, name in [(res_coco, "COCO 'person' (before)"), (res_ft, "fine-tuned (after)")]:
    plt.plot(r["rec"], r["prec"], label=f"{name}: AP@0.5={r['AP50']:.3f}")
plt.xlabel("recall"); plt.ylabel("precision"); plt.xlim(0, 1.02); plt.ylim(0, 1.02); plt.grid(alpha=.3)
plt.title("Precision-recall @ IoU 0.5 (50 held-out images)"); plt.legend(loc="lower left")
plt.tight_layout(); plt.savefig(OUT_DIR / "det_pr_curves.png", dpi=90); plt.show()

**Optional cross-check with `pycocotools`** (the official COCO evaluator). Numbers differ slightly from ours because
COCO uses 101-point interpolation and its own area rules, but they should agree closely — a good habit when you write
your own metric code.

In [ ]:
try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import contextlib, io

    def coco_ap(res):
        images, anns, dets = [], [], []
        for i, (p, g) in enumerate(zip(res["preds"], res["gts"])):
            images.append({"id": i, "width": 1000, "height": 1000})
            for b in g.tolist():
                anns.append({"id": len(anns) + 1, "image_id": i, "category_id": 1, "iscrowd": 0,
                             "bbox": [b[0], b[1], b[2] - b[0], b[3] - b[1]], "area": (b[2] - b[0]) * (b[3] - b[1])})
            for b, s in zip(p["boxes"].tolist(), p["scores"].tolist()):
                dets.append({"image_id": i, "category_id": 1, "score": s, "bbox": [b[0], b[1], b[2] - b[0], b[3] - b[1]]})
        gt = COCO(); gt.dataset = {"images": images, "annotations": anns, "categories": [{"id": 1, "name": "person"}]}
        with contextlib.redirect_stdout(io.StringIO()):
            gt.createIndex(); dt = gt.loadRes(dets)
            E = COCOeval(gt, dt, "bbox"); E.evaluate(); E.accumulate(); E.summarize()
        return E.stats[1], E.stats[0]      # AP50, AP50:95

    for name, r in [("COCO 'person' (before)", res_coco), ("fine-tuned (after)", res_ft)]:
        a50, a = coco_ap(r)
        print(f"{name:24s} ours AP50={r['AP50']:.3f} AP50:95={r['AP50_95']:.3f} | pycocotools AP50={a50:.3f} AP50:95={a:.3f}")
except ImportError:
    print("pycocotools not installed - skipping cross-check")

## 7. Visual check on held-out images — before vs after
Green = ground truth, red = prediction with score ≥ 0.5. Look for: missed people, duplicate boxes, boxes on
background people that Penn-Fudan did not annotate, and box tightness.

In [ ]:
SCORE_THR = 0.5
show = [0, 5, 10, 17, 25, 33]
fig, axes = plt.subplots(2, len(show), figsize=(3.3 * len(show), 8))
for col, k in enumerate(show):
    img, tgt = ds_test[k]
    for row, (r, name) in enumerate([(res_coco, "before (COCO person)"), (res_ft, "after fine-tuning")]):
        p = r["preds"][k]; keep = p["scores"] >= SCORE_THR
        ax = axes[row, col]
        draw_boxes(ax, img, tgt["boxes"], color="lime", lw=3)
        for b, s in zip(p["boxes"][keep].tolist(), p["scores"][keep].tolist()):
            ax.add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, edgecolor="red", linewidth=1.5))
            ax.text(b[0], b[1] - 2, f"{s:.2f}", color="white", fontsize=7, bbox=dict(facecolor="red", alpha=.7, pad=1, edgecolor="none"))
        ax.set_title(f"{name}\nGT={len(tgt['boxes'])} pred={int(keep.sum())}", fontsize=9)
plt.tight_layout(); plt.savefig(OUT_DIR / "det_predictions_before_after.png", dpi=80); plt.show()

## 8. Wrap-up — what to emphasise

* **Transfer learning is the default for detection:** 120 images + ~10 CPU-minutes gives a usable single-class detector
  because backbone, FPN, RPN and box head were all pre-trained on COCO.
* **Evaluate with the right metric:** accuracy is meaningless for detection; AP combines localisation (IoU) and ranking
  (confidence) and punishes duplicates.
* **Pre-trained vs fine-tuned — read the table honestly:** for "person" the COCO model is already excellent (AP@0.5 ≈ 0.98),
  so AP barely moves. The gain shows up at the operating threshold (compare the FP@0.5 column and the visual grid: the
  fine-tuned model follows Penn-Fudan's annotation convention). The spectacular number is the random 2-class head going
  from AP ≈ 0.02 to ≈ 0.96 after one CPU epoch — that is the power of transfer learning.
* **Next:** `02_detection_optimization.ipynb` — the same checkpoint, made faster on CPU. The checkpoint is in
  `checkpoints/fasterrcnn_mbv3_pennfudan.pth`.

In [ ]:
print(f"Checkpoint for the optimisation demo: {CKPT_PATH.resolve()}")
print(f"AP@0.5: COCO person = {res_coco['AP50']:.3f} | random new head = {res_random_head['AP50']:.3f} | fine-tuned = {res_ft['AP50']:.3f}")
print(f"False positives at score>=0.5: COCO person = {res_coco['op']['FP']} | fine-tuned = {res_ft['op']['FP']}")